In [8]:
import random
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [9]:
ledger_df = pd.read_csv('/content/ledger.csv')
ledger_df.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card


In [10]:
gateway_df = pd.read_csv('/content/gateway.csv')
gateway_df.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI


In [11]:
#Transactions missing in gateway

ledger_ids = set(ledger_df['transaction_id'])
gateway_ids = set(gateway_df['transaction_id'])

missing_transactions = ledger_ids - gateway_ids
#print(missing_transactions)

#Details of missing transactions
missing_transactions_gateway = ledger_df[ledger_df['transaction_id'].isin(missing_transactions)]
missing_transactions_gateway.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
3,R004,2026-03-02,M003,2100.0,success,Card
9,R010,2026-03-05,M004,2500.0,success,Wallet


In [12]:
# Export the dataframe to a CSV file
missing_transactions_gateway.to_csv('missing_gateway_transactions.csv', index=False)

print("File exported successfully!")

File exported successfully!


In [13]:
#Transactions missing in ledger

missing_transactions = gateway_ids - ledger_ids
#print(missing_transactions)

#Details of missing transactions
missing_transactions_ledger = ledger_df[ledger_df['transaction_id'].isin(missing_transactions)]
missing_transactions_ledger['issue_type'] = 'Missing in Ledger'
missing_transactions_ledger.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,issue_type


In [14]:
# Export the dataframe to a CSV file
missing_transactions_ledger.to_csv('missing_gateway_ledger.csv', index=False)

print("File exported successfully!")

File exported successfully!


In [15]:
#Amount mismatches

common_ids = list(ledger_ids & gateway_ids)

ledger_common = ledger_df[ledger_df['transaction_id'].isin(common_ids)].copy()
gateway_common = gateway_df[gateway_df['transaction_id'].isin(common_ids)].copy()

comparision = ledger_common.merge(gateway_common, on='transaction_id', suffixes=('_ledger', '_gateway'))

comparision['amount_difference'] = comparision['amount_usd_ledger'] - comparision['amount_usd_gateway']
comparision.head(100)

#DETAILS

amount_mismatch = comparision[comparision['amount_difference']!=0]
if len(amount_mismatch) > 0:
  print(f"Total amount mismatch = {amount_mismatch['amount_difference'].sum()}")

amount_mismatch.head(100)

Total amount mismatch = -10.0


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,amount_difference
1,R002,2026-03-01,M002,850.0,success,Card,2026-03-01,M002,900.0,success,Card,-50.0
6,R008,2026-03-04,M001,640.0,success,Card,2026-03-04,M001,600.0,success,Card,40.0


In [16]:
# Export the dataframe to a CSV file
amount_mismatch.to_csv('amount_mismatch.csv', index=False)

print("File exported successfully!")

File exported successfully!


In [17]:
#Status mismatch

status_mismatch = comparision[comparision['status_ledger'] != comparision['status_gateway']]
status_mismatch.head(100)

,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,amount_difference
3,R005,2026-03-03,M004,7200.0,success,Card,2026-03-03,M004,7200.0,failed,Card,0.0


In [18]:
# Export the dataframe to a CSV file
status_mismatch.to_csv('status_mismatch.csv', index=False)

print("File exported successfully!")

File exported successfully!


In [19]:
#Report creation

amount_mismatch_report = amount_mismatch[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'amount_difference']].copy()
amount_mismatch_report['issue_type'] = 'Amount Mismatch'
amount_mismatch_report.columns = ['transaction_id', 'amount_ledger', 'amount_gateway', 'amount_difference', 'issue_type']
amount_mismatch_report.head()

status_mismatch_report = status_mismatch[['transaction_id', 'status_ledger', 'status_gateway']].copy()
status_mismatch_report['issue_type'] = 'Status Mismatch'
status_mismatch_report.head()

missing_gateway_report = missing_transactions_gateway[['transaction_id', 'amount_usd']].copy()
missing_gateway_report['gateway_amount'] = 0
missing_gateway_report['difference'] = missing_gateway_report['amount_usd']
missing_gateway_report['issue_type'] = 'MISSING_IN_GATEWAY'
missing_gateway_report.columns = ['transaction_id', 'amount_ledger', 'amount_gateway', 'amount_difference', 'issue_type']
missing_gateway_report.head()


,transaction_id,amount_ledger,amount_gateway,amount_difference,issue_type
3,R004,2100.0,0,2100.0,MISSING_IN_GATEWAY
9,R010,2500.0,0,2500.0,MISSING_IN_GATEWAY


In [20]:
#Combining all reports

reconciliation_report = pd.concat([
    missing_gateway_report,
    amount_mismatch_report,
    status_mismatch_report
], ignore_index=True)

print(f"\n=== RECONCILIATION REPORT ===")
print(f"Total issues found: {len(reconciliation_report)}")

# Count by issue type
print(f"Count of issues by type")
print(reconciliation_report['issue_type'].value_counts())


=== RECONCILIATION REPORT ===
Total issues found: 5
Count of issues by type
issue_type
MISSING_IN_GATEWAY    2
Amount Mismatch       2
Status Mismatch       1
Name: count, dtype: int64


In [21]:
reconciliation_report.head()

,transaction_id,amount_ledger,amount_gateway,amount_difference,issue_type,status_ledger,status_gateway
0,R004,2100.0,0.0,2100.0,MISSING_IN_GATEWAY,NaN,NaN
1,R010,2500.0,0.0,2500.0,MISSING_IN_GATEWAY,NaN,NaN
2,R002,850.0,900.0,-50.0,Amount Mismatch,NaN,NaN
3,R008,640.0,600.0,40.0,Amount Mismatch,NaN,NaN
4,R005,NaN,NaN,NaN,Status Mismatch,success,failed


In [22]:
# Export the dataframe to a CSV file
reconciliation_report.to_csv('reconciliation_report.csv', index=False)

print("File exported successfully!")

File exported successfully!


In [23]:
import requests
import json
import pandas as pd
from pandas import json_normalize

In [24]:
response = {
  "generated_at": "2026-03-07T10:00:00Z",
  "source": "QuickPay Settlement API",
  "batches": [
    {
      "batch_id": "B001",
      "merchant": {
        "merchant_id": "M001",
        "merchant_name": "Alpha Mart",
        "region": "APAC"
      },
      "settlements": [
        {
          "settlement_id": "S001",
          "amount_usd": 1520.5,
          "status": "settled",
          "processed_at": "2026-03-07T08:10:00Z",
          "bank": {
            "name": "Bank A",
            "country": "IN"
          }
        },
        {
          "settlement_id": "S002",
          "amount_usd": 980.0,
          "status": "pending",
          "processed_at": "2026-03-07T08:45:00Z",
          "bank": {
            "name": "Bank A",
            "country": "IN"
          }
        },
        {
          "settlement_id": "S003",
          "amount_usd": 640.0,
          "status": "settled",
          "processed_at": "2026-03-07T09:15:00Z",
          "bank": {
            "name": "Bank B",
            "country": "SG"
          }
        }
      ]
    },
    {
      "batch_id": "B002",
      "merchant": {
        "merchant_id": "M004",
        "merchant_name": "Delta Travels",
        "region": "US"
      },
      "settlements": [
        {
          "settlement_id": "S004",
          "amount_usd": 2100.0,
          "status": "settled",
          "processed_at": "2026-03-07T08:20:00Z",
          "bank": {
            "name": "Bank C",
            "country": "US"
          }
        },
        {
          "settlement_id": "S005",
          "amount_usd": 500.0,
          "status": "failed",
          "processed_at": "2026-03-07T08:50:00Z",
          "bank": {
            "name": "Bank C",
            "country": "US"
          }
        },
        {
          "settlement_id": "S006",
          "amount_usd": 7200.0,
          "status": "settled",
          "processed_at": "2026-03-07T09:30:00Z",
          "bank": {
            "name": "Bank C",
            "country": "US"
          }
        }
      ]
    }
  ]
}

In [25]:
data_frame = pd.DataFrame(response)

data_frame.head()

#still shows nested data

,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"{'batch_id': 'B001', 'merchant': {'merchant_id..."
1,2026-03-07T10:00:00Z,QuickPay Settlement API,"{'batch_id': 'B002', 'merchant': {'merchant_id..."


In [26]:
response_list = response["batches"]
print(response_list)

import pandas as pd
from pandas import json_normalize

# Create the flattened DataFrame
response_df = json_normalize(
    response,
    record_path=['batches', 'settlements'],
    meta=[
        'generated_at',
        'source',
        ['batches', 'batch_id'],
        ['batches', 'merchant', 'merchant_name'],
        ['batches', 'merchant', 'region']
    ],
)

# Display the result
response_df.head()

[{'batch_id': 'B001', 'merchant': {'merchant_id': 'M001', 'merchant_name': 'Alpha Mart', 'region': 'APAC'}, 'settlements': [{'settlement_id': 'S001', 'amount_usd': 1520.5, 'status': 'settled', 'processed_at': '2026-03-07T08:10:00Z', 'bank': {'name': 'Bank A', 'country': 'IN'}}, {'settlement_id': 'S002', 'amount_usd': 980.0, 'status': 'pending', 'processed_at': '2026-03-07T08:45:00Z', 'bank': {'name': 'Bank A', 'country': 'IN'}}, {'settlement_id': 'S003', 'amount_usd': 640.0, 'status': 'settled', 'processed_at': '2026-03-07T09:15:00Z', 'bank': {'name': 'Bank B', 'country': 'SG'}}]}, {'batch_id': 'B002', 'merchant': {'merchant_id': 'M004', 'merchant_name': 'Delta Travels', 'region': 'US'}, 'settlements': [{'settlement_id': 'S004', 'amount_usd': 2100.0, 'status': 'settled', 'processed_at': '2026-03-07T08:20:00Z', 'bank': {'name': 'Bank C', 'country': 'US'}}, {'settlement_id': 'S005', 'amount_usd': 500.0, 'status': 'failed', 'processed_at': '2026-03-07T08:50:00Z', 'bank': {'name': 'Bank C'

,settlement_id,amount_usd,status,processed_at,bank.name,bank.country,generated_at,source,batches.batch_id,batches.merchant.merchant_name,batches.merchant.region
0,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN,2026-03-07T10:00:00Z,QuickPay Settlement API,B001,Alpha Mart,APAC
1,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN,2026-03-07T10:00:00Z,QuickPay Settlement API,B001,Alpha Mart,APAC
2,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG,2026-03-07T10:00:00Z,QuickPay Settlement API,B001,Alpha Mart,APAC
3,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US,2026-03-07T10:00:00Z,QuickPay Settlement API,B002,Delta Travels,US
4,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US,2026-03-07T10:00:00Z,QuickPay Settlement API,B002,Delta Travels,US


In [27]:
response_df.columns = ['settlement_id', 'amount', 'status', 'processed_at',
                            'bank_name', 'bank_country','generated_at', 'source',  'batch_id', 'merchant_name', 'region']
response_df.head()

,settlement_id,amount,status,processed_at,bank_name,bank_country,generated_at,source,batch_id,merchant_name,region
0,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN,2026-03-07T10:00:00Z,QuickPay Settlement API,B001,Alpha Mart,APAC
1,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN,2026-03-07T10:00:00Z,QuickPay Settlement API,B001,Alpha Mart,APAC
2,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG,2026-03-07T10:00:00Z,QuickPay Settlement API,B001,Alpha Mart,APAC
3,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US,2026-03-07T10:00:00Z,QuickPay Settlement API,B002,Delta Travels,US
4,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US,2026-03-07T10:00:00Z,QuickPay Settlement API,B002,Delta Travels,US


In [28]:
# Export the dataframe to a CSV file
response_df.to_csv('api_normalised.csv', index=False)

print("File exported successfully!")

File exported successfully!
